# MAI645 Colab Runner

This notebook mounts Google Drive, installs the repository requirements, and runs the required MAI645 preprocessing, training, and evaluation entrypoints for positional, Euler angle, and quaternion representations.

The assignment handout does not include the concrete original GitHub repository URL or BVH dataset path in the downloaded materials. Update every `TODO` path below before setting `DRY_RUN = False`.

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ModuleNotFoundError:
    print("Not running inside Google Colab; skipping Drive mount.")

In [ ]:
from pathlib import Path

# TODO: update these paths for your Google Drive layout.
PROJECT_ROOT = Path("/content/drive/MyDrive/MAI645/mai645-character-motion-prediction")
GITHUB_REPO_URL = "TODO_paste_your_github_repo_url_if_PROJECT_ROOT_is_not_already_present"
ASSIGNMENT_REPO_PATH = Path("/content/drive/MyDrive/MAI645/TODO_original_assignment_repo")
BVH_DATASET_PATH = Path("/content/drive/MyDrive/MAI645/TODO_bvh_dataset_folder")
DRIVE_RUN_ROOT = Path("/content/drive/MyDrive/MAI645/runs")

# Keep True until the TODO paths and code/mai645_runner.py adapter sections are filled in.
DRY_RUN = True

PROCESSED_ROOT = DRIVE_RUN_ROOT / "processed"
MODEL_ROOT = DRIVE_RUN_ROOT / "models"
OUTPUT_ROOT = DRIVE_RUN_ROOT / "outputs"

PROJECT_ROOT

In [ ]:
import os
import subprocess
import sys

def is_todo(value):
    text = str(value)
    return any(token in text for token in ("TODO", "REPLACE_ME", "UNKNOWN", "<"))

if not PROJECT_ROOT.exists():
    if is_todo(GITHUB_REPO_URL):
        raise FileNotFoundError(
            f"PROJECT_ROOT does not exist: {PROJECT_ROOT}\n"
            "Clone or copy this repository there, or set GITHUB_REPO_URL above."
        )
    PROJECT_ROOT.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "clone", GITHUB_REPO_URL, str(PROJECT_ROOT)], check=True)

os.chdir(PROJECT_ROOT)
print(f"Working directory: {Path.cwd()}")

In [ ]:
env = os.environ.copy()
env["MAI645_DRIVE_ROOT"] = str(DRIVE_RUN_ROOT)
subprocess.run(["bash", "scripts/setup_colab.sh"], check=True, env=env)

In [ ]:
if DRY_RUN:
    print("DRY_RUN is enabled. Commands will validate wiring but will not require real TODO paths.")
else:
    required_paths = {
        "ASSIGNMENT_REPO_PATH": ASSIGNMENT_REPO_PATH,
        "BVH_DATASET_PATH": BVH_DATASET_PATH,
    }
    for name, path in required_paths.items():
        if is_todo(path):
            raise ValueError(f"TODO: replace {name}: {path}")
        if not path.exists():
            raise FileNotFoundError(f"{name} does not exist: {path}")

for path in (PROCESSED_ROOT, MODEL_ROOT, OUTPUT_ROOT):
    path.mkdir(parents=True, exist_ok=True)

print(f"Processed arrays: {PROCESSED_ROOT}")
print(f"Models: {MODEL_ROOT}")
print(f"Outputs: {OUTPUT_ROOT}")

In [ ]:
def run_stage(command):
    command = list(command)
    if DRY_RUN:
        command.append("--dry-run")
    print("\n$ " + " ".join(str(part) for part in command))
    subprocess.run(command, check=True)

PREPROCESS_SCRIPTS = {
    "pos": "code/generate_training_pos_data.py",
    "euler": "code/generate_training_euler_data.py",
    "quad": "code/generate_training_quad_data.py",
}

TRAIN_SCRIPTS = {
    "pos": "code/pytorch_train_pos_aclstm.py",
    "euler": "code/pytorch_train_euler_aclstm.py",
    "quad": "code/pytorch_train_quad_aclstm.py",
}

EVAL_SCRIPTS = {
    "pos": "code/synthise_pos_motion.py",
    "euler": "code/synthise_euler_motion.py",
    "quad": "code/synthise_quad_motion.py",
}

## 1. Preprocessing

Converts BVH files into the three required character motion representations. Replace the TODO adapter in `code/mai645_runner.py` with the original repository's BVH loading/conversion logic.

In [ ]:
for representation, script in PREPROCESS_SCRIPTS.items():
    run_stage([
        sys.executable,
        script,
        "--assignment-repo", str(ASSIGNMENT_REPO_PATH),
        "--bvh-dir", str(BVH_DATASET_PATH),
        "--output-dir", str(PROCESSED_ROOT / representation),
    ])

## 2. Training

Trains one AC-LSTM model per representation. Adjust the epochs, batch size, sequence length, and learning rate as needed for final experiments.

In [ ]:
for representation, script in TRAIN_SCRIPTS.items():
    run_stage([
        sys.executable,
        script,
        "--assignment-repo", str(ASSIGNMENT_REPO_PATH),
        "--processed-dir", str(PROCESSED_ROOT / representation),
        "--model-dir", str(MODEL_ROOT / representation),
        "--epochs", "10",
        "--batch-size", "32",
        "--sequence-length", "120",
        "--learning-rate", "0.001",
    ])

## 3. Evaluation and BVH Synthesis

Loads each trained model, computes quantitative error for generated motion, and writes generated BVH output for qualitative review.

In [ ]:
MODEL_PATHS = {
    "pos": MODEL_ROOT / "pos" / "TODO_pos_model.pt",
    "euler": MODEL_ROOT / "euler" / "TODO_euler_model.pt",
    "quad": MODEL_ROOT / "quad" / "TODO_quad_model.pt",
}

for representation, script in EVAL_SCRIPTS.items():
    run_stage([
        sys.executable,
        script,
        "--assignment-repo", str(ASSIGNMENT_REPO_PATH),
        "--processed-dir", str(PROCESSED_ROOT / representation),
        "--model-path", str(MODEL_PATHS[representation]),
        "--output-dir", str(OUTPUT_ROOT / representation),
        "--seed-frames", "20",
        "--generated-frames", "400",
    ])